In [ ]:
# Import packages and modules
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow_decision_forests as tfdf

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [ ]:
# Check the version of TensorFlow Decision Forests
print("Found TensorFlow Decision Forests v" + tfdf.__version__)

In [ ]:
# train/validation/test = 70/10/20
datasetPath = 'dataset/firstorder/kernel5-radius5/300.dataset.csv'
dataset = pd.read_csv(datasetPath)

train_data, temp_data = train_test_split(
    dataset, test_size=0.3, random_state=42
 )
validation_data, test_data = train_test_split(
    temp_data, test_size=2/3, random_state=42
 )

In [ ]:
# Convert the dataset into a TensorFlow dataset.
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_data, label="label"
)         
val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    validation_data, label="label"
)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_data, label="label"
)

In [ ]:
import keras

In [ ]:
%%time

# Train an SVM model with class weight.
svm_model = SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")
svm_model.fit(train_data.drop(columns=["label"]), train_data["label"])

In [ ]:
# Evaluate the model with sklearn
from sklearn.metrics import accuracy_score

X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
y_pred = svm_model.predict(X_val)

accuracy = accuracy_score(y_true, y_pred)
print(f"accuracy: {accuracy:.4f}")

In [ ]:
# Model Summary (sklearn style)
print(svm_model)
print(f"kernel: {svm_model.kernel}")
print(f"C: {svm_model.C}")
print(f"gamma: {svm_model.gamma}")

In [ ]:
# Model features
feature_names = train_data.drop(columns=["label"]).columns.tolist()
print("features:")
print(feature_names)

In [ ]:
# Feature importance (not available for SVC with RBF kernel)
print("SVC with RBF kernel does not provide feature importances.")

In [ ]:
# Model self evaluation (sklearn info)
print("number of support vectors:", svm_model.support_vectors_.shape[0])
print("support vectors per class:", svm_model.n_support_)

In [ ]:
# Training logs (not available for sklearn SVC)
print("Training logs are not available for sklearn SVC.")

Calculate the score of our hold-out validation dataset

In [ ]:
X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]

from sklearn.metrics import roc_auc_score
ROC_AUC = roc_auc_score(y_true, pos_probs)
print("The ROC AUC score is %.5f" % ROC_AUC )

In [ ]:
# Compute binary classification metrics with sklearn
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    log_loss,
    brier_score_loss,
 )

# Get predictions (probabilities or class labels)
X_val = test_data.drop(columns=["label"])
y_true = test_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]
y_pred = (pos_probs >= 0.5).astype(int)

# Core metrics
metrics = {}
metrics["accuracy"] = accuracy_score(y_true, y_pred)
metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
metrics["mcc"] = matthews_corrcoef(y_true, y_pred)

# Probabilistic metrics
metrics["roc_auc"] = roc_auc_score(y_true, pos_probs)
metrics["pr_auc"] = average_precision_score(y_true, pos_probs)
metrics["log_loss"] = log_loss(y_true, pos_probs, labels=[0,1])
metrics["brier_score"] = brier_score_loss(y_true, pos_probs)

# Confusion matrix and detailed report
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
report = classification_report(y_true, y_pred, digits=4)

print("Sklearn binary metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
print("\nConfusion matrix:\n", cm)
print("\nClassification report:\n", report)